In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


In [ ]:
def impute_knn(df, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df_copy = df.copy()
    df_copy['Throughput'] = imputer.fit_transform(df_copy[['Throughput']])
    return df_copy

# def impute_rolling_median(df, window_size=3):
#     df_copy = df.copy()
#     previous_na_count = df_copy['Throughput'].isna().sum()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).median())
#         current_na_count = df_copy['Throughput'].isna().sum()
#         if current_na_count >= previous_na_count:
#             break  # No progress made, so exit the loop
#         previous_na_count = current_na_count
#     # global_median = df_copy['Throughput'].median()
#     # df_copy['Throughput'] = df_copy['Throughput'].fillna(global_median)
#     return df_copy

# def impute_rolling_average(df, window_size=3):
#     df_copy = df.copy()
#     previous_na_count = df_copy['Throughput'].isna().sum()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).mean())
#         current_na_count = df_copy['Throughput'].isna().sum()
#         if current_na_count >= previous_na_count:
#             break  # No progress made, so exit the loop
#         previous_na_count = current_na_count
#     # global_mean = df_copy['Throughput'].mean()
#     # df_copy['Throughput'] = df_copy['Throughput'].fillna(global_mean)
#     return df_copy

# def impute_rolling_median(df, window_size=3):
#     df_copy = df.copy()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).median())
#     return df_copy

# def impute_rolling_average(df, window_size=3):
#     df_copy = df.copy()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).mean())
#     return df_copy

def impute_rolling_median(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).median())
    global_median = df['Throughput'].median()
    df['Throughput'] = df['Throughput'].fillna(global_median)
    return df

def impute_rolling_average(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).mean())
    global_mean = df['Throughput'].mean()
    df['Throughput'] = df['Throughput'].fillna(global_mean)
    return df

def linear_interpolation(df, limit_direction='both', method='linear'):
    df_imputed = df.interpolate(method=method, limit_direction=limit_direction)

    df['Throughput'] = df['Throughput'].fillna(df_imputed['Throughput'])

    return df


In [ ]:
def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"CSV file '{caminho_arquivo_csv}' generated!")
    return df1

def matriz(path):
    df = pd.read_csv(path)
    df = outlier_removal(df, 'Throughput')
    df_datetime = df.copy()
    df_datetime.drop(columns=['Throughput'], inplace = True)
    df['Throughput'] = df['Throughput'].replace(-1, np.nan)
    
    Throughput = df['Throughput'].values
    num_dados = len(Throughput)
    num_colunas = num_dados // 28
    matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_original = pd.DataFrame(matriz)
    df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
    Throughput_=df_interpolado.values
    matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_interpolado= pd.DataFrame(matriz_interpolado)
    mask = np.isnan(matriz_original.values)
    matriz_mascara = pd.DataFrame(mask)
    return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max

In [ ]:
def get_dataset_path_list(diretory):
    dataset_path_list = []
    for file in os.listdir(diretory):
        path = os.path.join(diretory, file)
        dataset_path_list.append(path)
    return dataset_path_list

In [ ]:


def apply_basic_imputations(source_path, destination_path, csv_path_list):
    for caminho in csv_path_list:
        df = pd.read_csv(caminho)
        
        if df.shape[0] < 28:
            print(f'The {caminho} file does not have sufficient quantity of lines for imputation (28)')
            continue

        df = outlier_removal(df, 'Throughput')
        
        df_knn = impute_knn(df.copy())
        df_rolling_median = impute_rolling_median(df.copy())
        df_rolling_average = impute_rolling_average(df.copy())
        df_interpolation = linear_interpolation(df.copy())
        
        # Create output directories if they don't exist
        techniques = ['knn', 'mediana-movel', 'media-movel', 'interpolacao-linear']
        for technique in techniques:
            technique_dir = os.path.join(destination_path, technique)
            if not os.path.exists(technique_dir):
                os.makedirs(technique_dir)
                print(f"Saving path for {technique} created.")
        
        # Construct output file paths
        relative_path = os.path.relpath(caminho, source_path)
        output_knn = os.path.join(destination_path, 'knn', relative_path)
        output_median = os.path.join(destination_path, 'mediana-movel', relative_path)
        output_average = os.path.join(destination_path, 'media-movel', relative_path)
        output_interpolation = os.path.join(destination_path, 'interpolacao-linear', relative_path)
        
        # Ensure the output directories exist
        os.makedirs(os.path.dirname(output_knn), exist_ok=True)
        os.makedirs(os.path.dirname(output_median), exist_ok=True)
        os.makedirs(os.path.dirname(output_average), exist_ok=True)
        os.makedirs(os.path.dirname(output_interpolation), exist_ok=True)
        
        # Save the DataFrames to CSV
        df_knn.to_csv(output_knn, index=False)
        df_rolling_median.to_csv(output_median, index=False)
        df_rolling_average.to_csv(output_average, index=False)
        df_interpolation.to_csv(output_interpolation, index=False)
        
        print(f"Processed file: {caminho}")


#adicionar uma verificação para caso o tamanho do arquivo seja menor que 28 -> remover porque se nao fica só interpolacao linear?
def apply_svd_imputation(destination_path, csv_path_list):
    
    resultados = {}

    for caminho_csv in csv_path_list:

        df = pd.read_csv(caminho_csv)

        if (df.shape[0] < 28):
            print(f'The {caminho_csv} file does not have sufficient quantity of lines for imputation (28)')
            continue

        nome_arquivo = os.path.basename(caminho_csv)
        
        resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
        df_matriz, df_mask, df_interpolado, df_datetime = matriz(caminho_csv)
        resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

        A_anterior = df_interpolado.values.copy()
        rmse = float('inf') 
        max_iter = 300
        n_iter = 0


        while rmse >= 1e-3 and n_iter<=max_iter:  
            U, S, Vt = decomposicao_svd(df_interpolado)
            variabilidade = np.cumsum(S**2) / np.sum(S**2)

            porcentagem_variabilidade = 0.95
            r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
            
            #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

            U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
            S_reduzido_matriz = np.diag(S_reduzido)

            A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
            A_aproximada_df = pd.DataFrame(A_aproximada)

            df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
            
            resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

            # Atualiza df_interpolado para a próxima iteração
            df_interpolado = df_matriz_preenchida.values
            
            # Calcular o RMSE entre a matriz atual e a anterior
            rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

            # Atualiza A_anterior para a próxima comparação
            A_anterior = A_aproximada.copy()

            n_iter +=1

        # print(f'RMSE na iteração atual: {rmse}')
        # print(f'Finalizando processamento para {caminho_csv}')

        svd = matriztodf(resultados[nome_arquivo]["svd_final"])
        interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
        mask = matriztodf(df_mask)
        dfs_reshaped = []
        dfs_reshaped.append(interpolacao) #0
        dfs_reshaped.append(svd) #1
        dfs_reshaped.append(mask) #2
        # Chama a função para plotar os dados
        # plot_imputed_data(dfs_reshaped)
        gerar_arq_csv(df_datetime, dfs_reshaped[1], destination_path, nome_arquivo)
        

In [ ]:
def apply_all_imputations(source_path, destination_path):
    csv_path_list = get_dataset_path_list(source_path)
    apply_basic_imputations(source_path, destination_path, csv_path_list)
    apply_svd_imputation(destination_path, csv_path_list)

In [ ]:
source_path = '../datasets/choosen-best-svd/'
destination_path = '../datasets/imputed-choosen-best-svd/'


In [ ]:
apply_all_imputations(source_path, destination_path)

In [ ]:
# source_path_longest_interval = '../datasets/treated longest interval with failures/'
# destination_path_longest_interval = '../datasets/imputed-treated-longest-interval-with-failures/'
# apply_all_imputations(source_path_longest_interval, destination_path_longest_interval)